# Regression Metrics — From Residuals Upward

Every metric starts from the same place: **the residual** (how wrong each prediction is).

We build each metric step by step — each one fixing a problem with the previous one.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

## Dataset — 5 Students

In [2]:
df = pd.DataFrame({
    'Student': ['A', 'B', 'C', 'D', 'E'],
    'Hours':   [1, 2, 3, 4, 5],
    'Score':   [52, 58, 65, 71, 79]
})

X = df[['Hours']]
y = df['Score']

model = LinearRegression().fit(X, y)
print(f"Model: Score = {model.intercept_:.1f} + {model.coef_[0]:.1f} × Hours")
df

Model: Score = 44.9 + 6.7 × Hours


,Student,Hours,Score
0,A,1,52
1,B,2,58
2,C,3,65
3,D,4,71
4,E,5,79


## Step 0 — Residuals

**Residual = actual y − predicted ŷ**

This is the raw error for each data point. Every metric is built from these.

In [3]:
y_pred = model.predict(X)
residuals = y - y_pred

df['ŷ (predicted)'] = y_pred.round(1)
df['Residual (y − ŷ)'] = residuals.round(2)
df

,Student,Hours,Score,ŷ (predicted),Residual (y − ŷ)
0,A,1,52,51.6,0.4
1,B,2,58,58.3,-0.3
2,C,3,65,65.0,0.0
3,D,4,71,71.7,-0.7
4,E,5,79,78.4,0.6


## Step 1 — Mean Error (ME)

Most obvious idea: just average the residuals.

**Problem:** Positive and negative errors cancel → always ≈ 0. Useless.

In [4]:
me = np.mean(y - y_pred)
print(f"Sum of residuals : {(y - y_pred).sum():.4f}")
print(f"Mean Error (ME)  : {me:.4f}")
print()
print("→ Errors cancel out. ME = 0 even though individual predictions are wrong.")
print("→ Fix: remove the sign")

Sum of residuals : -0.0000
Mean Error (ME)  : -0.0000

→ Errors cancel out. ME = 0 even though individual predictions are wrong.
→ Fix: remove the sign


## Step 2 — MAE (Mean Absolute Error)

Take **absolute value** of each residual before averaging — removes the sign.

In [5]:
abs_errors = np.abs(y - y_pred)

df_mae = df[['Student', 'Score', 'ŷ (predicted)', 'Residual (y − ŷ)']].copy()
df_mae['|Residual|'] = abs_errors.round(2)
print(df_mae.to_string(index=False))

mae_manual = abs_errors.mean()
mae_sklearn = mean_absolute_error(y, y_pred)

print(f"\nMAE (manual) : {mae_manual:.4f}")
print(f"MAE (sklearn): {mae_sklearn:.4f}")
print(f"\n→ On average the model is off by {mae_sklearn:.2f} score points")
print("→ Fix: squaring penalises large errors more than small ones")

Student  Score  ŷ (predicted)  Residual (y − ŷ)  |Residual|
      A     52           51.6               0.4         0.4
      B     58           58.3              -0.3         0.3
      C     65           65.0               0.0         0.0
      D     71           71.7              -0.7         0.7
      E     79           78.4               0.6         0.6

MAE (manual) : 0.4000
MAE (sklearn): 0.4000

→ On average the model is off by 0.40 score points
→ Fix: squaring penalises large errors more than small ones


## Step 3 — SSE (Sum of Squared Errors)

**Square** each residual and sum them — removes sign AND penalises large errors more.

This is what OLS minimises to find the best line.

In [6]:
squared_errors = (y - y_pred) ** 2

df_sse = df[['Student', 'Residual (y − ŷ)']].copy()
df_sse['Residual²'] = squared_errors.round(4)
print(df_sse.to_string(index=False))

sse = squared_errors.sum()
print(f"\nSSE = {sse:.4f}")
print("→ OLS minimises this to find β₀ and β₁")
print("→ Fix: SSE grows with dataset size — divide by n to normalise")

Student  Residual (y − ŷ)  Residual²
      A               0.4       0.16
      B              -0.3       0.09
      C               0.0       0.00
      D              -0.7       0.49
      E               0.6       0.36

SSE = 1.1000
→ OLS minimises this to find β₀ and β₁
→ Fix: SSE grows with dataset size — divide by n to normalise


## Step 4 — MSE (Mean Squared Error)

Divide SSE by n — now comparable across different dataset sizes.

**Note:** SSE = MSE × n  (NOT MSE²)

In [7]:
n = len(y)

mse_manual  = sse / n
mse_sklearn = mean_squared_error(y, y_pred)
sse_from_mse = mse_sklearn * n       # MSE × n — NOT MSE²

print(f"n              : {n}")
print(f"SSE            : {sse:.4f}")
print(f"MSE (manual)   : {mse_manual:.4f}   (SSE / n = {sse:.4f} / {n})")
print(f"MSE (sklearn)  : {mse_sklearn:.4f}")
print(f"SSE from MSE   : {sse_from_mse:.4f}   (MSE × n — NOT MSE²)")
print("→ Fix: units are score² — hard to interpret. Take square root")

n              : 5
SSE            : 1.1000
MSE (manual)   : 0.2200   (SSE / n = 1.1000 / 5)
MSE (sklearn)  : 0.2200
SSE from MSE   : 1.1000   (MSE × n — NOT MSE²)
→ Fix: units are score² — hard to interpret. Take square root


## Step 5 — RMSE (Root Mean Squared Error)

Take √MSE — back in original units (score points).

**Thumb rule:** RMSE < std dev of y → model beats naïve mean prediction.

In [8]:
rmse   = np.sqrt(mse_sklearn)
y_std  = y.std()

print(f"RMSE           : {rmse:.4f} score points")
print(f"Std dev of y   : {y_std:.4f} score points")
print(f"RMSE < std dev : {rmse:.2f} < {y_std:.2f} → {rmse < y_std}")
print()
print("→ Model is much better than just predicting the mean every time ✓")
print("→ Fix: 0.47 out of what? Doesn't tell us % error. Use MAPE")

RMSE           : 0.4690 score points
Std dev of y   : 10.6066 score points
RMSE < std dev : 0.47 < 10.61 → True

→ Model is much better than just predicting the mean every time ✓
→ Fix: 0.47 out of what? Doesn't tell us % error. Use MAPE


## Step 6 — MAPE (Mean Absolute Percentage Error)

Divide each absolute error by the actual value — expresses error as % of actual.

**Note:** sklearn returns a fraction (0–1), multiply by 100 for %.

In [9]:
pct_errors = np.abs(y - y_pred) / y * 100

df_mape = df[['Student', 'Score', 'ŷ (predicted)', 'Residual (y − ŷ)']].copy()
df_mape['|Residual|'] = abs_errors.round(2)
df_mape['% Error'] = pct_errors.round(2).values
print(df_mape.to_string(index=False))

mape_manual  = pct_errors.mean()
mape_sklearn = mean_absolute_percentage_error(y, y_pred) * 100   # ← multiply by 100

print(f"\nMAPE (manual) : {mape_manual:.4f}%")
print(f"MAPE (sklearn): {mape_sklearn:.4f}%")
print(f"\n→ Model is off by {mape_sklearn:.2f}% on average — excellent (< 10% is good)")

Student  Score  ŷ (predicted)  Residual (y − ŷ)  |Residual|  % Error
      A     52           51.6               0.4         0.4     0.77
      B     58           58.3              -0.3         0.3     0.52
      C     65           65.0               0.0         0.0     0.00
      D     71           71.7              -0.7         0.7     0.99
      E     79           78.4               0.6         0.6     0.76

MAPE (manual) : 0.6064%
MAPE (sklearn): 0.6064%

→ Model is off by 0.61% on average — excellent (< 10% is good)


## Step 7 — SST (Total Sum of Squares)

Shift question: **how much variance is there in y to explain?**

SST = error if you predicted ȳ (the mean) for everyone — the dumb baseline.

In [10]:
y_mean = y.mean()
baseline_errors = (y - y_mean) ** 2

df_sst = pd.DataFrame({
    'Student': df['Student'],
    'y':       y.values,
    'ȳ':       y_mean,
    'y − ȳ':   (y - y_mean).values,
    '(y − ȳ)²': baseline_errors.round(2).values
})
print(df_sst.to_string(index=False))

sst = baseline_errors.sum()
print(f"\nȳ (mean of y) : {y_mean:.1f}")
print(f"SST           : {sst:.2f}")
print("→ If you predict ȳ=65 for every student, your total squared error is 450")
print("→ Your model's SSE must be less than this to be useful")

Student  y    ȳ  y − ȳ  (y − ȳ)²
      A 52 65.0  -13.0     169.0
      B 58 65.0   -7.0      49.0
      C 65 65.0    0.0       0.0
      D 71 65.0    6.0      36.0
      E 79 65.0   14.0     196.0

ȳ (mean of y) : 65.0
SST           : 450.00
→ If you predict ȳ=65 for every student, your total squared error is 450
→ Your model's SSE must be less than this to be useful


## Step 8 — SSR (Sum of Squares Regression)

How much variance did the model **explain**?

SSR = how far predictions move away from ȳ toward actual values.

In [11]:
explained = (y_pred - y_mean) ** 2

df_ssr = pd.DataFrame({
    'Student':   df['Student'],
    'ŷ':         y_pred.round(1),
    'ȳ':         y_mean,
    'ŷ − ȳ':     (y_pred - y_mean).round(1),
    '(ŷ − ȳ)²':  explained.round(2)
})
print(df_ssr.to_string(index=False))

ssr = explained.sum()
print(f"\nSSR : {ssr:.2f}")
print(f"SSE : {sse:.2f}")
print(f"SST : {sst:.2f}")
print(f"\nSSR + SSE = {ssr:.2f} + {sse:.2f} = {ssr + sse:.2f}  ({'✓' if abs((ssr + sse) - sst) < 0.01 else '✗'} equals SST)")

Student    ŷ    ȳ  ŷ − ȳ  (ŷ − ȳ)²
      A 51.6 65.0  -13.4    179.56
      B 58.3 65.0   -6.7     44.89
      C 65.0 65.0    0.0      0.00
      D 71.7 65.0    6.7     44.89
      E 78.4 65.0   13.4    179.56

SSR : 448.90
SSE : 1.10
SST : 450.00

SSR + SSE = 448.90 + 1.10 = 450.00  (✓ equals SST)


## Step 9 — R² (R-Squared)

What **fraction** of total variance did the model explain?

R² = SSR / SST = 1 − SSE / SST

In [12]:
r2_manual  = ssr / sst
r2_manual2 = 1 - (sse / sst)
r2_sklearn = r2_score(y, y_pred)

print(f"R² = SSR / SST       = {ssr:.2f} / {sst:.2f} = {r2_manual:.4f}")
print(f"R² = 1 − (SSE / SST) = 1 − ({sse:.2f} / {sst:.2f}) = {r2_manual2:.4f}")
print(f"R² (sklearn)         = {r2_sklearn:.4f}")
print(f"\n→ Model explains {r2_sklearn*100:.1f}% of the variance in scores")
print(f"→ Remaining {(1-r2_sklearn)*100:.1f}% is unexplained noise")

R² = SSR / SST       = 448.90 / 450.00 = 0.9976
R² = 1 − (SSE / SST) = 1 − (1.10 / 450.00) = 0.9976
R² (sklearn)         = 0.9976

→ Model explains 99.8% of the variance in scores
→ Remaining 0.2% is unexplained noise


## Step 10 — Adjusted R²

R² always increases when you add features — even useless ones.

Adjusted R² penalises adding features that don't help.

No sklearn built-in — calculate manually.

In [13]:
p = X.shape[1]   # number of features
adj_r2 = 1 - (1 - r2_sklearn) * (n - 1) / (n - p - 1)

print(f"n (samples)  : {n}")
print(f"p (features) : {p}")
print(f"R²           : {r2_sklearn:.4f}")
print(f"Adjusted R²  : {adj_r2:.4f}")
print()
print("→ With only 1 feature, R² and Adj R² are close")
print("→ Adj R² matters more when comparing models with different numbers of features")

n (samples)  : 5
p (features) : 1
R²           : 0.9976
Adjusted R²  : 0.9967

→ With only 1 feature, R² and Adj R² are close
→ Adj R² matters more when comparing models with different numbers of features


## All Metrics Together

In [14]:
summary = pd.DataFrame({
    'Metric':  ['ME', 'MAE', 'SSE', 'MSE', 'RMSE', 'MAPE', 'SST', 'SSR', 'R²', 'Adj R²'],
    'Value':   [
        round(me, 4),
        round(mae_sklearn, 4),
        round(sse, 4),
        round(mse_sklearn, 4),
        round(rmse, 4),
        f"{mape_sklearn:.2f}%",
        round(sst, 2),
        round(ssr, 2),
        round(r2_sklearn, 4),
        round(adj_r2, 4)
    ],
    'Interpretation': [
        'Useless — errors cancel',
        'Off by 0.40 score pts on avg',
        'Total squared error; OLS minimises this',
        'Normalised squared error',
        'Off by 0.47 score pts (same units as y)',
        'Off by 0.61% of actual value',
        'Total variance in y (baseline)',
        'Variance explained by model',
        '99.8% of variance explained',
        'R² penalised for number of features'
    ]
})
summary

,Metric,Value,Interpretation
0,ME,-0.0,Useless — errors cancel
1,MAE,0.4,Off by 0.40 score pts on avg
2,SSE,1.1,Total squared error; OLS minimises this
3,MSE,0.22,Normalised squared error
4,RMSE,0.469,Off by 0.47 score pts (same units as y)
5,MAPE,0.61%,Off by 0.61% of actual value
6,SST,450.0,Total variance in y (baseline)
7,SSR,448.9,Variance explained by model
8,R²,0.9976,99.8% of variance explained
9,Adj R²,0.9967,R² penalised for number of features
